### Install requirements

In [1]:
!pip install transformers datasets torch sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 38.7 MB/s eta 0:00:00


### Set up torch

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load the dataset

In [3]:
from datasets import load_from_disk, Dataset
italian_english_dataset = load_from_disk("/kaggle/input/xnli-50k/italian_english_dataset")
italian_english_dataset = Dataset.from_dict(italian_english_dataset.to_dict())

### Import the model and tokenizer

In [4]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
import numpy as np

### Load the tokenizer

In [5]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large", model_max_length=256)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

### Tokenize the dataset

In [6]:
def tokenize_fn(batch):
    # Create input strings with language markers
    premises_en = [f"[LANG_EN] {p['en']}" for p in batch["premise"]]
    hypotheses_en = [f"[LANG_EN] {h['en']}" for h in batch["hypothesis"]]
    premises_it = [f"[LANG_IT] {p['it']}" for p in batch["premise"]]
    hypotheses_it = [f"[LANG_IT] {h['it']}" for h in batch["hypothesis"]]

    # Change to 255 later
    max_half_length = 127
    
    # Tokenize both languages
    tokens_en = tokenizer(
        premises_en, hypotheses_en,
        truncation=True, padding="max_length", max_length=max_half_length
    )
    tokens_it = tokenizer(
        premises_it, hypotheses_it,
        truncation=True, padding="max_length", max_length=max_half_length
    )

    # Combine the results
    input_ids = []
    attention_masks = []

    for i in range(len(batch["premise"])):
        input_ids.append(
            [tokenizer.cls_token_id] + tokens_en["input_ids"][i][1:] + 
            [tokenizer.sep_token_id] + tokens_it["input_ids"][i][1:] + 
            [tokenizer.sep_token_id]
        )
        attention_masks.append(
            [1] + tokens_en["attention_mask"][i][1:] + 
            [1] + tokens_it["attention_mask"][i][1:] + 
            [1]
        )
    
    # Return the tokenized data
    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": batch["label"]   
    }

# Apply tokenization
tokenized_dataset = italian_english_dataset.map(tokenize_fn, batched=True, batch_size=32, num_proc=4)

Map (num_proc=4):   0%|          | 0/50000 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

In [7]:
max_lengths = [len(seq) for seq in tokenized_dataset["input_ids"]]
print(f"Min length: {min(max_lengths)}, Max length: {max(max_lengths)}")

Min length: 255, Max length: 255


### Remove unnecessary columns

In [8]:
tokenized_dataset.remove_columns(["premise", "hypothesis", "label"])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50000
})

### Load the model

In [9]:
model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-large", num_labels=3)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Split the dataset into train and test sets

In [10]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

### Take a sample from the dataset

In [11]:
sample_dataset = tokenized_dataset.shuffle(seed=42).select(range(100))
split_sample_dataset = sample_dataset.train_test_split(test_size=0.2)

### Set up the hyperparameters and train

In [12]:
# Change into the actual dataset
dataset = split_sample_dataset

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,              
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                       
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=lambda p: {"f1": f1_score(p.predictions.argmax(axis=1), p.label_ids, average="weighted")},
)

trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=10, training_loss=1.1160003662109375, metrics={'train_runtime': 27.7184, 'train_samples_per_second': 5.772, 'train_steps_per_second': 0.361, 'total_flos': 74263531435200.0, 'train_loss': 1.1160003662109375, 'epoch': 2.0})